### Step 1: Install necesscary packages

In [1]:
!pip install matplotlib
!pip install torch numpy transformers datasets tiktoken wandb tqdm

In [1]:
# Step 0: Setup

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone repository
print("\n📥 Cloning repository...")
!git clone https://github.com/zhouyuan888888/NanoGPT-Math.git

# 3. Install dependencies
print("\n📦 Installing dependencies...")
!pip install -q torch numpy transformers datasets tiktoken wandb tqdm matplotlib gdown

# 4. Download pretrained model
print("\n⬇️ Downloading pretrained model...")
import os
os.chdir('/content/NanoGPT-Math')
!gdown 1gIZw-HAB-tHtEYCmNugwlIV7R3WsgjjZ -O ./sft/gpt.pt

# 5. Setup paths and imports
import sys
os.chdir('/content/NanoGPT-Math/dpo')
sys.path.append(os.path.abspath(".."))

import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
from tqdm import tqdm
import json
import matplotlib.pyplot as plt

# 6. Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 5
batch_size = 64
max_length = 64
max_new_tokens = 200
temperature = 0.8
top_k = 200

# 7. Load tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]

def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return ''.join([itos[i] for i in l])

# 8. Verification
print("\n" + "=" * 60)
print("✓ SETUP COMPLETE!")
print("=" * 60)
print(f"✓ Device: {device}")
if not torch.cuda.is_available():
    print("⚠️  WARNING: GPU not available! Training will be slow.")
    print("   Go to: Runtime → Change runtime type → GPU")
print(f"✓ Vocabulary size: {len(stoi)}")
print(f"✓ Current directory: {os.getcwd()}")
print("✓ Ready to proceed with tasks!")
print("=" * 60)

Mounted at /content/drive

📥 Cloning repository...
Cloning into 'NanoGPT-Math'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 34 (delta 9), reused 7 (delta 7), pack-reused 17 (from 1)
Receiving objects: 100% (34/34), 13.82 KiB | 13.82 MiB/s, done.
Resolving deltas: 100% (9/9), done.

📦 Installing dependencies...

⬇️ Downloading pretrained model...
Downloading...
From (original): https://drive.google.com/uc?id=1gIZw-HAB-tHtEYCmNugwlIV7R3WsgjjZ
From (redirected): https://drive.google.com/uc?id=1gIZw-HAB-tHtEYCmNugwlIV7R3WsgjjZ&confirm=t&uuid=6e61d6c2-ec5d-48fb-bc9a-575e0f269083
To: /content/NanoGPT-Math/sft/gpt.pt
100% 106M/106M [00:00<00:00, 160MB/s] 

✓ SETUP COMPLETE!
✓ Device: cuda
✓ Vocabulary size: 74
✓ Current directory: /content/NanoGPT-Math/dpo
✓ Ready to proceed with tasks!


### Step 2: Package imports and configuration

In [2]:
import sys
import os
sys.path.append(os.path.abspath(".."))
#os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 5
batch_size = 64
max_length = 64
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

print(device)
print(stoi)


cuda
{'\n': 0, ' ': 1, "'": 2, '*': 3, '+': 4, ',': 5, '-': 6, '.': 7, '/': 8, '=': 9, '?': 10, '’': 11, '0': 12, '1': 13, '2': 14, '3': 15, '4': 16, '5': 17, '6': 18, '7': 19, '8': 20, '9': 21, 'A': 22, 'B': 23, 'C': 24, 'D': 25, 'E': 26, 'F': 27, 'G': 28, 'H': 29, 'I': 30, 'J': 31, 'K': 32, 'L': 33, 'M': 34, 'N': 35, 'O': 36, 'P': 37, 'Q': 38, 'R': 39, 'S': 40, 'T': 41, 'U': 42, 'V': 43, 'W': 44, 'X': 45, 'Y': 46, 'Z': 47, 'a': 48, 'b': 49, 'c': 50, 'd': 51, 'e': 52, 'f': 53, 'g': 54, 'h': 55, 'i': 56, 'j': 57, 'k': 58, 'l': 59, 'm': 60, 'n': 61, 'o': 62, 'p': 63, 'q': 64, 'r': 65, 's': 66, 't': 67, 'u': 68, 'v': 69, 'w': 70, 'x': 71, 'y': 72, 'z': 73}


We can observe that the tokenizer works on the character-level. As such, complex reasoning chains might not be useful for improving arithmatic understanding of the model.

### Step 3: Define helper functions

In [ ]:
# Compute log probabilities of sequences
def compute_logprob(model, input_ids):

    inputs = input_ids[:, :-1] # Input tokens: all except the last one 
    targets = input_ids[:, 1:] # Target tokens (to be predicted): all except the first one
    logits, _ = model(inputs, full_seq=True)
    
    B, T, V = logits.size()

    # Flatten the logits and targets for loss computation
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)

    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)

    # Create attention mask to ignore padding tokens (indicated by 0) for loss calculation
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1) # exclude padding
    return -loss 

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))


def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])

    # In the code here, only full batches are yielded, incomplete ones are skipped
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

In [4]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device)

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

### Task 1: Generate dataset
We are generating 200k negative and positive pairs for our training data for the subsequent finetuning. The data count is evenly split between two types of question: direct arithmetic and algebraic question to solve for unknown variable x. For the latter, we also vary the position of the unknown variable x randomly to obtain a more diverse set of questions. For each question, the numbers are drawn randomly from [1,100] and the operators are also randomly drawn.

For the positive responses, we model them after the provided examples in providing concise explanation. For the algebraic questions, we simply reverse the operation in our explanation. We found that this concise reasoning chain works better than complex ones generated by conventional LLM models like ChatGPT, which may provide a longer explanation for each question. Not only does this save time in generating positive response (avoid the need to query another LLM), the character-level tokenisation of the model also limits the model understanding of complex reasoning chains as it lacks the vocabulary expressivity unlike larger models like ChatGPT. 

For the negative responses, we also previous tried to use the generated responses from the provided NanoGPT model, which often produce responses with explanation like "Sorry, I do not know" but with a bit of variation in terms of misspellings and abbreviations. As we want to scale our dataset to 200k, we find that it suffices to simply hardcode the above explanation as the negative examples for our model training.

In [ ]:
import random

N = 250000

def make_direct():
    op = random.choice(["+", "-", "*", "/"])
    a, b = random.randint(1, 100), random.randint(1, 100)

    if op == "+":
        ans = a + b
        explanation = f"{a}+{b} equals {ans}"
    elif op == "-":
        ans = a - b
        explanation = f"{a}-{b} equals {ans}"
    elif op == "*":
        ans = a * b
        explanation = f"{a}*{b} equals {ans}"
    else:
        a = a * b
        ans = a // b
        explanation = f"{a}/{b} equals {ans}"

    q_str = f"{a}{op}{b}=?"
    return f"{q_str} The answer is {ans} because {explanation}.", f"{q_str} Sorry, I do not know."

def make_solve_x():
    op = random.choice(["+", "-", "*", "/"])

    ans, b = random.randint(1, 100), random.randint(1, 100)

    if op == "+":
        rhs = ans + b
        # Randomly decide the position of x
        if random.randint(0,1) == 0:
            q_str = f"x+{b}={rhs},x=?"
        else:
            q_str = f"{b}+x={rhs},x=?"
        explanation = f"{rhs}-{b} equals to {ans}"

    elif op == "-":
        # Randomly decide the position of x
        if random.randint(0,1) == 0:
            rhs = ans - b
            q_str = f"x-{b}={rhs},x=?"
            explanation = f"{rhs}+{b} equals to {ans}"
        else:
            rhs = b - ans
            q_str = f"{b}-x={rhs},x=?"
            if rhs > 0:
                explanation = f"{b}-{rhs} equals to {ans}"
            else:
                explanation = f"{b}+{-rhs} equals to {ans}"

    elif op == "*":
        rhs = ans * b
        # Randomly decide the position of x
        if random.randint(0,1) == 0:
            q_str = f"x*{b}={rhs},x=?"
        else:
            q_str = f"{b}*x={rhs},x=?"
        explanation = f"{rhs}/{b} equals to {ans}"
    else:
        rhs = ans
        # Randomly decide the position of x
        if random.randint(0,1) == 0:
            ans = ans * b
            rhs = ans // b
            q_str = f"x/{b}={rhs},x=?"
            explanation = f"{rhs}*{b} equals to {ans}"
        else:
            b = ans * b
            rhs = b // ans
            q_str = f"{b}/x={rhs},x=?"
            explanation = f"{b}/{rhs} equals to {ans}"
    return f"{q_str} The answer is {ans} because {explanation}.", f"{q_str} Sorry, I do not know."

neg_ans, pos_ans = [], []
json_file = "../data/pos_neg_pairs.json"


for _ in range(N//2):
    pos, neg = make_direct()
    neg_ans.append(neg)
    pos_ans.append(pos)

    pos, neg = make_solve_x()
    neg_ans.append(neg)
    pos_ans.append(pos)

data = [{"negative": n, "positive": p} for n, p in zip(neg_ans, pos_ans)]

with open(json_file, "w") as f:
    json.dump(data, f, indent=4)

print(f"Saved {len(data)} QA pairs to {json_file}")


Saved 250000 QA pairs to ../data/pos_neg_pairs.json


### Step 5: Load Data (**students are required to complete this part!**)

In [7]:
# Load data from ./data/pos_neg_pairs.json and split into train/val/test
import random
json_file = "../data/pos_neg_pairs.json"
with open(json_file) as f:
    all_lines = json.load(f)

# Shuffle and split: 80% train, 10% val, 10% test
random.seed(42)
random.shuffle(all_lines)
n_total = len(all_lines)
n_train = int(0.8 * n_total)
n_val = int(0.1 * n_total)

train_lines = all_lines[:n_train]
val_lines = all_lines[n_train:n_train + n_val]
test_lines = all_lines[n_train + n_val:]

print(f"Total samples: {n_total}")
print(f"Train: {len(train_lines)}, Val: {len(val_lines)}, Test: {len(test_lines)}")
print("\nExample from training set:")
print(train_lines[0])

Total samples: 250000
Train: 200000, Val: 25000, Test: 25000

Example from training set:
{'negative': 'x/8=75,x=? Sorry, I do not know.', 'positive': 'x/8=75,x=? The answer is 600 because 75*8 equals to 600.'}


### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

Our optimiser choice is AdamW (Adam with decoupled weight decay). Unlike Adam, AdamW applies weight decay in a separate gradient update instead of combining it into the usual loss function. This way, it ensures that the effect of weight regularisation is independent of the bias correction term (whose value changes over time in training) used in Adam, allowing regularisation effect to be constant throughout the training. Comapred to Adam, this approach is found to have improvements in model generalisation. 

In [8]:
# recommend to use the AdamW optimizer
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

epochs = 20
num_batches = len(train_lines) // batch_size
total_steps = num_batches * epochs

optim = AdamW(gpt.parameters(), lr=base_lr)
scheduler = get_linear_schedule_with_warmup(
    optim,
    num_warmup_steps=int(0.15 * total_steps),
    num_training_steps=total_steps
)

print(f"Number of training batches per epoch: {num_batches}")

Number of training batches per epoch: 3125


In [ ]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
ref = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
ref.load_state_dict(state_dict)
ref.to(device)

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

Over here, we are instantiating another copy of the supervised finetuned model to be used as a reference model for the DPO loss function. 

### Step 7: Begin training (**students are required to complete this part!**)
Our training approach differs from the provided code snippet in the loss function used. Instead of simply minimising the negative log-likelihood difference of the probability of current model producing positive response over the negative response (provided by the code snippet), we also find obtain the same likelihood difference produced by the reference model, and this will be incorporated into the loss function to act as weight regularisation to align our finetuned model such that it does not deviate too much away from the reference model. Note that minimising negative log-likelihood is equivalent to maximising log-likelihood.

In [ ]:
# Helper function to evaluate on validation set
def evaluate_validation():
    gpt.eval()
    val_loss = 0
    val_batches = 0
    with torch.no_grad():
        for neg_tensor, pos_tensor in get_batches(val_lines, batch_size):
            neg_logprob = compute_logprob(gpt, neg_tensor)
            pos_logprob = compute_logprob(gpt, pos_tensor)
            loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1
            val_loss += loss.item()
            val_batches += 1
    gpt.train()
    return val_loss / max(val_batches, 1)

# Training loop with validation-based early stopping
gpt.train()
ref.eval()  # Keep reference model in eval mode

# Define early stopping parameters to trigger early stopping after 3 epochs of no improvement on validation loss
best_val_loss = float('inf') # Initialise to infinity to ensure
patience = 3
min_improvement = 1e-3
wait = 0

for epoch in range(epochs):
    epoch_loss = 0
    pbar = tqdm(get_batches(train_lines, batch_size))
    for step, (neg_tensor, pos_tensor) in enumerate(pbar):
        # Zero our the gradients for every new gradient calculation step
        optim.zero_grad()

        # Policy model logprobs
        neg_logprob = compute_logprob(gpt, neg_tensor)
        pos_logprob = compute_logprob(gpt, pos_tensor)

        # Reference model logprobs
        # Apply torch.no_grad() to avoid computing gradients for the reference model
        with torch.no_grad():
            ref_neg_logprob = compute_logprob(ref, neg_tensor)
            ref_pos_logprob = compute_logprob(ref, pos_tensor)

        # Calculate the differences in log probabilities
        pi_diff = pos_logprob - neg_logprob
        ref_diff = ref_pos_logprob - ref_neg_logprob

        # Compute the DPO loss incorporating reference model log probabilities and an additional term to encourage higher log probabilities for positive examples
        loss = -F.logsigmoid((pi_diff - ref_diff) / beta).mean() - pos_logprob.mean() * 0.1

        # Backpropagate the loss
        loss.backward()
        
        optim.step()
        scheduler.step()
        epoch_loss += loss.item()

    train_loss = epoch_loss / num_batches
    val_loss = evaluate_validation()
    print(f"Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    if best_val_loss - val_loss > min_improvement:
        best_val_loss = val_loss
        wait = 0
        ckpt_path = f"./dpo.pt"
        torch.save({
            "model_state_dict": gpt.state_dict(),
            "model_args": ckpt['model_args'],
        }, ckpt_path)
        print(f"Saved checkpoint to {ckpt_path}")
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping!")
            break

3125it [02:50, 18.38it/s]


Epoch 1 | Train Loss: 0.1544 | Val Loss: 0.0667
Saved checkpoint to ./dpo.pt


3125it [02:48, 18.52it/s]


Epoch 2 | Train Loss: 0.0451 | Val Loss: 0.0300
Saved checkpoint to ./dpo.pt


3125it [02:48, 18.54it/s]


...

Epoch 9 | Train Loss: 0.0208 | Val Loss: 0.0203


3125it [02:48, 18.51it/s]


Epoch 10 | Train Loss: 0.0206 | Val Loss: 0.0201


3125it [02:48, 18.50it/s]


Epoch 11 | Train Loss: 0.0204 | Val Loss: 0.0200
Early stopping!


### Step 8: Begin testing (**students are required to complete this part!**)

In [12]:
# Load the fine-tuned model
ckpt_path = "./dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).cuda()
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]
with torch.no_grad():
    for prompt in test_set:
        prompt_ids = encode(prompt)
        x = (torch.tensor(prompt_ids, dtype=torch.long, device=device)[None, ...])
        y = gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        print('------------------------')
        print(decode(y[0][0].tolist()))
        print('------------------------')


------------------------
17+19=? The answer is 36 because 17+19 equals 36.
------------------------
------------------------
3*17=? The answer is 51 because 3*17 equals 51.
------------------------
------------------------
72/4=? The answer is 18 because 72/4 equals 18.
------------------------
------------------------
72-x=34,x=? The answer is 38 because 72-34 equals to 38.
------------------------
------------------------
x*11=44,x=? The answer is 4 because 44/11 equals to 4.
------------------------
------------------------
3*17=? The answer is 51 because 3*17 equals 51.
------------------------
------------------------
72/4=? The answer is 18 because 72/4 equals 18.
------------------------
------------------------
72-x=34,x=? The answer is 38 because 72-34 equals to 38.
------------------------


In [13]:
### Step 9: Automated evaluation on test set

import re

def extract_answer(text):
    """Extract numerical answer from generated text using regex"""
    # Look for "answer is X" pattern
    match = re.search(r'answer is\s+(-?\d+)', text.lower())
    if match:
        return int(match.group(1))
    return None

def extract_ground_truth(positive_text):
    """Extract ground truth answer from positive example"""
    match = re.search(r'answer is\s+(-?\d+)', positive_text.lower())
    if match:
        return int(match.group(1))
    return None

# Sample 1000 test examples (or all if less than 1000)
test_sample = random.sample(test_lines, min(1000, len(test_lines)))

# Load the fine-tuned model
ckpt_path = "./dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt_eval = GPT(gptconf).to(device)
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k, v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt_eval.load_state_dict(state_dict)

# Evaluate
gpt_eval.eval()
correct = 0
total = 0
errors = []

print("Evaluating on test set...")
with torch.no_grad():
    for item in tqdm(test_sample):
        # Extract question from positive example (before "The answer is")
        pos_text = item['positive']
        question = pos_text.split('The answer is')[0].strip()

        # Get ground truth
        gt_answer = extract_ground_truth(pos_text)
        if gt_answer is None:
            continue

        # Generate prediction
        prompt_ids = encode(question)
        x = torch.tensor(prompt_ids, dtype=torch.long, device=device)[None, ...]
        y = gpt_eval.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        generated_text = decode(y[0][0].tolist())

        # Extract predicted answer
        pred_answer = extract_answer(generated_text)

        if pred_answer is not None:
            total += 1
            if pred_answer == gt_answer:
                correct += 1
            else:
                errors.append({
                    'question': question,
                    'predicted': pred_answer,
                    'ground_truth': gt_answer,
                    'generated': generated_text
                })

# Print results
accuracy = (correct / total * 100) if total > 0 else 0
print(f"\n{'='*60}")
print(f"Test Set Evaluation Results")
print(f"{'='*60}")
print(f"Total evaluated: {total}")
print(f"Correct: {correct}")
print(f"Accuracy: {accuracy:.2f}%")
print(f"{'='*60}")

# Show a few error examples
if errors:
    print(f"\nShowing up to 5 error examples:")
    for i, error in enumerate(errors[:5], 1):
        print(f"\n--- Error {i} ---")
        print(f"Q: {error['question']}")

Evaluating on test set...


100%|██████████| 1000/1000 [03:30<00:00,  4.74it/s]


Test Set Evaluation Results
Total evaluated: 1000
Correct: 795
Accuracy: 79.50%

Showing up to 5 error examples:

--- Error 1 ---
Q: x/42=89,x=?
Predicted: 3768 | Ground Truth: 3738

--- Error 2 ---
Q: 81*26=?
Predicted: 2026 | Ground Truth: 2106

--- Error 3 ---
Q: 62*25=?
Predicted: 1500 | Ground Truth: 1550

--- Error 4 ---
Q: 72*x=6768,x=?
Predicted: 84 | Ground Truth: 94

--- Error 5 ---
Q: x*72=2520,x=?
Predicted: 30 | Ground Truth: 35
